In [ ]:
import pandas as pd

import numpy as np
import os

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import torch
torch.manual_seed(100)

import random
random.seed(15)

import numpy as np
np.random.seed(30)

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
DIR_INPUTS = './data_raw/'
DIR_RUNTIME_DATA = './data_runtime/'
DIR_RUNTIME_RESULTS = './results_runtime/'
TUNE = True
TRANS_ONLY=False
TUNE_LR = False
N_TRIALS_TUNE = 64 #100
MAX_EPOCH_TUNE = 100
MAX_EPOCH_TRAIN = 500
PATIENCE = 50
WEIGHTED = True

torch.set_float32_matmul_precision('medium')

In [ ]:
# Clean up the runtime data folder
import shutil
if TUNE:
    for root, dirs, files in os.walk(DIR_RUNTIME_DATA):
        for f in files:
            os.unlink(os.path.join(root, f))
        for d in dirs:
            shutil.rmtree(os.path.join(root, d))

    for root, dirs, files in os.walk(DIR_RUNTIME_RESULTS):
        for f in files:
            os.unlink(os.path.join(root, f))
        for d in dirs:
            shutil.rmtree(os.path.join(root, d))

### Create train-validation and test sets (into csvs)

In [ ]:
df_features = pd.read_csv(os.path.join(DIR_INPUTS,'df_features.csv'), index_col=0)
df_features.index = df_features.index.rename('subject')
df_features = df_features.reset_index()
df_state_history_sampled_max = pd.read_csv(os.path.join(DIR_INPUTS,'df_state_history_sampled_max.csv'),  index_col=0)


In [ ]:
MAX_STATES = df_state_history_sampled_max['state'].max()
N_TIME = df_state_history_sampled_max['time'].max()

In [ ]:
N_TIME

In [ ]:
MAX_STATES

In [ ]:
df_features.head()

In [ ]:
df_state_history_sampled_max.head()

In [ ]:
df_state_history_sampled_max.max()

In [ ]:
subj_train = df_features['subject'][::df_features['subject'].size//1000]
subj_train_tune = subj_train.sample(n=500)
subj_val = df_features['subject'].loc[~df_features['subject'].isin(subj_train)].sample(n=500)
subj_test = df_features['subject'].loc[
    ~(
        df_features['subject'].isin(subj_train) |
        df_features['subject'].isin(subj_val)
    )
]


for suffix, subj in [
    ('train_tune', subj_train_tune),
    ('train', subj_train),
    ('val', subj_val),
    ('test', subj_test)
]:
    print(f'n_subj in {suffix}:{len(subj)}')
    df_features_sub = df_features.loc[
        df_features['subject'].isin(subj),
        :
    ]
    df_features_sub.to_csv(os.path.join(DIR_RUNTIME_DATA, f'df_features_{suffix}.csv'))

    df_state_history_sampled_max_sub = df_state_history_sampled_max.loc[
        df_state_history_sampled_max['subject'].isin(subj),
        :
    ]
    df_state_history_sampled_max_sub.to_csv(os.path.join(DIR_RUNTIME_DATA, f'df_state_history_sampled_max_{suffix}.csv'))

del df_state_history_sampled_max_sub
del df_features_sub
del df_state_history_sampled_max
del df_features

## Load train val and test sets

In [ ]:
from torch.utils.data import DataLoader
from monotonic_nn_surv_surf.utils.datasets_def import DatasetFeatANDtgy

In [ ]:
ds_train = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_train.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_train.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    trans_only=TRANS_ONLY,
    weighted=WEIGHTED
)
loader_train = DataLoader(ds_train, batch_size=1000,shuffle=True)

ds_train_tune = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_train_tune.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_train_tune.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    trans_only=TRANS_ONLY,
    weighted=WEIGHTED
)
loader_train_tune = DataLoader(ds_train, batch_size=500,shuffle=True)

ds_val = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_val.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_val.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    trans_only=TRANS_ONLY,
    weighted=WEIGHTED
)
loader_val = DataLoader(ds_val, batch_size=1000)

ds_test = DatasetFeatANDtgy(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_test.csv'),
    path_state_history_max_grade=os.path.join(DIR_RUNTIME_DATA,'df_state_history_sampled_max_test.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
    trans_only=TRANS_ONLY,
    weighted=WEIGHTED
)
loader_test = DataLoader(ds_test, batch_size=1000)

In [ ]:
len(ds_train_tune)

In [ ]:
len(ds_train)

In [ ]:
len(ds_val)

## Train model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
import pytorch_lightning as pl

pl.seed_everything(seed=20)

In [ ]:
from monotonic_nn_surv_surf.utils.pl_model_wrapper import LitSurvSurf
from monotonic_nn_surv_surf.utils.surv_surf_latent import SurvSurfLatent, LatentFeatFC

#### Hyperparam

In [ ]:

def objective(trial):

    early_stop = pl.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE)
    
    n_monotone_layers = trial.suggest_int('n_monotone_layers', 4, 16)
    n_monoton_neurons = trial.suggest_int('n_monoton_neurons', 8, 64)
    n_feat_layers = trial.suggest_int('n_feat_layers', 4, 16)
    n_feat_neurons = trial.suggest_int('n_feat_neurons', 8, 64)
    p_dropout = trial.suggest_uniform('p_dropout', 0, 0.5)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)

    model = SurvSurfLatent(
        mono_net_sizes=[n_feat_neurons] + [n_monoton_neurons]*n_monotone_layers + [1],
        latent_feat_transformer=LatentFeatFC(
            input_size=3, 
            output_size=n_feat_neurons, 
            neurons_per_layer=(n_feat_layers-1)*[n_feat_neurons],
            dropout_p=p_dropout
        ),
    )
    
    model_lit = LitSurvSurf(model=model, lr=learning_rate)
     
    trainer = pl.Trainer(
        default_root_dir=DIR_RUNTIME_RESULTS,
        logger=False, 
        accelerator="gpu", 
        max_epochs=MAX_EPOCH_TUNE, 
        enable_progress_bar=False,
        check_val_every_n_epoch=1,
        callbacks = [early_stop]
    )      
    
    trainer.fit(
        model=model_lit, 
        train_dataloaders=loader_train_tune ,
        val_dataloaders=loader_val
    )
    
    score_val = trainer.test(model_lit, loader_val)[0]['test_loss']
    
    return score_val


In [ ]:
import optuna
if TUNE:
    sampler = optuna.samplers.TPESampler(seed=17)

    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS_TUNE)

### read best hyperparams and tune lr if necessary

In [ ]:
import json

if TUNE:
    print('best hyperparam and tuning performance:')
    print(study.best_trial.value)

    best_hyper_params = study.best_trial.params
    print(best_hyper_params)
    
    # Serializing json
    json_object = json.dumps(best_hyper_params, indent=4)
    
    # Writing to sample.json
    with open(os.path.join(DIR_RUNTIME_RESULTS,"best_hyper_params.json"), "w") as outfile:
        outfile.write(json_object)
else:
    # Opening JSON file
    with open(os.path.join(DIR_RUNTIME_RESULTS,"best_hyper_params.json"), 'r') as openfile:
    
        # Reading from json file
        best_hyper_params = json.load(openfile)

n_feat_neurons = best_hyper_params['n_feat_neurons']
n_monoton_neurons = best_hyper_params['n_monoton_neurons']
n_monotone_layers = best_hyper_params['n_monotone_layers']
n_feat_layers = best_hyper_params['n_feat_layers']
p_dropout = best_hyper_params['p_dropout']
learning_rate = best_hyper_params['learning_rate']


In [ ]:
best_hyper_params

In [ ]:
model = SurvSurfLatent(
    mono_net_sizes=[n_feat_neurons] + [n_monoton_neurons]*n_monotone_layers + [1],
    latent_feat_transformer=LatentFeatFC(
        input_size=3, 
        output_size=n_feat_neurons, 
        neurons_per_layer=(n_feat_layers-1)*[n_feat_neurons],
        dropout_p=p_dropout
    ),
)

model_lit = LitSurvSurf(model=model, lr=learning_rate, print_epoch=True)

In [ ]:
logger = pl.loggers.CSVLogger(save_dir=DIR_RUNTIME_RESULTS)

if not TUNE:
    if TUNE_LR:
        trainer = pl.Trainer(
                accelerator="gpu", 
                default_root_dir=DIR_RUNTIME_RESULTS,
                enable_progress_bar=False,
                logger=logger
            )
        tuner = pl.tuner.Tuner(trainer)

        # 3. Tune learning rate
        lr_finder = tuner.lr_find(
                model_lit, 
                train_dataloaders=loader_train,
                val_dataloaders=loader_val,
        )

        fig = lr_finder.plot(suggest=True)
        fig.show()
        new_lr = lr_finder.suggestion()

        # update hparams of the model
        model_lit.hparams.lr = new_lr
        print(f'found suggested lr at {new_lr}')

### actual train

In [ ]:
early_stop = pl.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE)
trainer = pl.Trainer(
    accelerator="gpu", 
    devices=1, 
    max_epochs=MAX_EPOCH_TRAIN, 
    logger=logger,
    enable_progress_bar=False,
    default_root_dir=DIR_RUNTIME_RESULTS,
    check_val_every_n_epoch=1,
    callbacks=[early_stop]
)
trainer.fit(
    model=model_lit, 
    train_dataloaders=loader_train ,
    val_dataloaders=loader_val
)

In [ ]:
trainer.test(model=model_lit, dataloaders=loader_train)

In [ ]:
trainer.test(model=model_lit, dataloaders=loader_val)

In [ ]:
trainer.test(model=model_lit, dataloaders=loader_test)

## Inspect learning curve

In [ ]:
dir_logs = os.path.join(DIR_RUNTIME_RESULTS, 'lightning_logs')
dir_logs

In [ ]:
latest_ver = sorted(os.listdir(dir_logs))[-1]
latest_ver

In [ ]:
epoch_metrics = pd.read_csv(os.path.join(dir_logs, f'{latest_ver}/metrics.csv'))
epoch_metrics.head(20)

In [ ]:
epoch_metrics = epoch_metrics.groupby('epoch').apply(
    lambda df: 
    pd.Series(
        [
            df['val_loss'].iloc[0],
            df['train_loss'].iloc[-1]
        ],
        index=['val_loss','train_loss']
    ), 
).reset_index()

In [ ]:
epoch_metrics['val_loss'].min()

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(
    epoch_metrics['epoch'],
    epoch_metrics['train_loss'],
    label='train_loss'
)
ax.plot(
    epoch_metrics['epoch'],
    epoch_metrics['val_loss'],
    label='val_loss'
)
ax.set(xlabel='epochs', ylabel='loss')
ax.legend()

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()